## Natural Language Processing Spring 2026
---
# Model Architecture (Live Coding)

# 📘 Notebook 2 — Model Architecture  *(Live Coding)*
**MiniMind Step-by-Step Series** | Milestone 2 of 6

> **Live Coding Session** — Infrastructure is pre-filled.
> Implement the **core building blocks** in the `# TODO` sections.

**Learning Objectives:**
- Define model hyperparameters with MiniMindConfig
- Implement RMSNorm (Root Mean Square Layer Normalization)
- Implement Rotary Position Embeddings (RoPE)
- Build Grouped-Query Attention (GQA)
- Implement the SwiGLU Feed-Forward Network
- Understand Mixture of Experts (MOE) routing
- Assemble a Transformer Block with pre-norm residual connections
- Wire everything into a complete MiniMind causal LM and count parameters


In [ ]:
# === Setup (recap from Notebook 1) ===
!pip install tokenizers torch transformers --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part A — Model Configuration

Before building layers, we define the model hyperparameters. MiniMind is a ~26M parameter transformer decoder with these key settings:

| Parameter | Value | Purpose |
|---|---|---|
| `hidden_size` | 768 | Dimension of hidden representations |
| `num_hidden_layers` | 8 | Number of transformer blocks |
| `num_attention_heads` | 8 | Number of query heads |
| `num_key_value_heads` | 4 | Fewer KV heads → Grouped-Query Attention |
| `head_dim` | 96 | Per-head dimension (hidden_size / num_attention_heads) |
| `intermediate_size` | 2048 | FFN inner dimension |
| `vocab_size` | 6400 | Our BPE vocabulary from Notebook 1 |
| `max_position_embeddings` | 32768 | Maximum sequence length |
| `rope_theta` | 1e6 | RoPE base frequency |

In [ ]:
# === Model Configuration ===
@dataclass
class MiniMindConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 8
    num_attention_heads: int = 8
    num_key_value_heads: int = 4       # GQA: fewer KV heads than Q heads
    head_dim: int = 96                  # hidden_size // num_attention_heads
    vocab_size: int = 6400
    intermediate_size: int = 2048
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1e6
    dropout: float = 0.0
    hidden_act: str = 'silu'
    flash_attn: bool = True
    bos_token_id: int = 1
    eos_token_id: int = 2
    use_moe: bool = False              # Enable Mixture of Experts
    num_experts: int = 4
    num_experts_per_tok: int = 1
    moe_intermediate_size: int = 2048
    norm_topk_prob: bool = True
    router_aux_loss_coef: float = 5e-4

config = MiniMindConfig()
print(f'Config: hidden_size={config.hidden_size}, layers={config.num_hidden_layers}')
print(f'  Q heads={config.num_attention_heads}, KV heads={config.num_key_value_heads}')
print(f'  head_dim={config.head_dim}, vocab_size={config.vocab_size}')
print(f'  intermediate_size={config.intermediate_size}')

## Part B — RMSNorm

Layer normalization stabilizes training by normalizing activations. **RMSNorm** (Root Mean Square Normalization) is a simplified variant used in modern transformers like LLaMA and MiniMind:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot \gamma$$

Unlike standard LayerNorm, RMSNorm **does not subtract the mean** — it only rescales by the root-mean-square. This makes it faster (fewer operations) while working just as well for transformers. The learnable parameter $\gamma$ (`weight`) allows the network to adjust the scale per dimension.

In [ ]:
# === RMSNorm ===
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def norm(self, x):
        # ============================================================
        # TODO: Implement RMS normalization
        #
        # Formula: x * rsqrt(mean(x^2) + eps)
        #
        # Steps:
        #   1. Square x element-wise: x.pow(2)
        #   2. Mean over last dim, keep dims: .mean(-1, keepdim=True)
        #   3. Add eps for numerical stability
        #   4. Take reciprocal square root: torch.rsqrt(...)
        #   5. Multiply x by the result
        #
        # Hint: torch.rsqrt(a) == 1 / sqrt(a)
        # ============================================================

        # YOUR CODE HERE
        pass

    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

print('RMSNorm defined \u2713')


In [ ]:
# === Test RMSNorm ===
rms = RMSNorm(dim=768)
x = torch.randn(2, 32, 768)
out = rms(x)

assert out.shape == x.shape, f'Shape mismatch: {out.shape}'
rms_vals = out.float().pow(2).mean(-1).sqrt()
print(f'Output shape: {out.shape}')
print(f'RMS values (should be ≈ 1): mean={rms_vals.mean():.4f}, std={rms_vals.std():.4f}')
print('✅ RMSNorm test passed')

## Part C — Rotary Position Embeddings (RoPE)

Transformers need to know token positions. **RoPE** encodes position by rotating query/key vectors in 2D subspaces. For each dimension pair $(d_{2i}, d_{2i+1})$ at position $m$:

$$\theta_i = \text{base}^{-2i/d}$$

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

**Key insight:** The dot product $q' \cdot k'$ depends only on the **relative** distance between positions, not absolute positions. This gives the model translation-invariant position awareness without learned position embeddings.

We precompute cosine and sine tables, then apply them during attention.

In [ ]:
# === Rotary Position Embeddings (RoPE) ===
def precompute_freqs_cis(dim, end=32768, rope_base=1e6):
    """Precompute cosine and sine tables for RoPE."""
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    # ============================================================
    # TODO: Build cosine and sine frequency tables
    #
    # Given per-dimension frequencies `freqs` (shape [dim//2]):
    #   1. Create position indices: t = torch.arange(end)
    #   2. Outer product: freqs = torch.outer(t, freqs).float()
    #      → shape (end, dim//2), each row = position × per-dim freq
    #   3. Duplicate across both halves (needed for rotate_half):
    #      freqs_cos = torch.cat([cos(freqs), cos(freqs)], dim=-1)
    #      freqs_sin = torch.cat([sin(freqs), sin(freqs)], dim=-1)
    #   4. Return freqs_cos, freqs_sin  (each shape: end × dim)
    # ============================================================

    # YOUR CODE HERE
    pass

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """Apply rotary embeddings to query and key tensors."""
    def rotate_half(x):
        # Split x in half along last dim and swap/negate halves
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)
    # ============================================================
    # TODO: Apply the rotation to q and k
    #
    # Steps:
    #   1. Add a batch dimension to cos and sin:
    #      cos = cos.unsqueeze(unsqueeze_dim)
    #      sin = sin.unsqueeze(unsqueeze_dim)
    #   2. Rotate: q_embed = (q * cos) + (rotate_half(q) * sin)
    #              k_embed = (k * cos) + (rotate_half(k) * sin)
    #   3. Cast back to original dtype: .to(q.dtype) / .to(k.dtype)
    #   4. Return q_embed, k_embed
    # ============================================================

    # YOUR CODE HERE
    pass

print('RoPE functions defined \u2713')


In [ ]:
# === Test RoPE ===
HEAD_DIM = 96
SEQ_LEN = 32
freqs_cos, freqs_sin = precompute_freqs_cis(dim=HEAD_DIM, end=SEQ_LEN)

assert freqs_cos.shape == (SEQ_LEN, HEAD_DIM), f'freqs_cos shape: {freqs_cos.shape}'
assert freqs_sin.shape == (SEQ_LEN, HEAD_DIM), f'freqs_sin shape: {freqs_sin.shape}'

# Test rotation preserves dot-product relative structure
q = torch.randn(2, SEQ_LEN, 8, HEAD_DIM)
k = torch.randn(2, SEQ_LEN, 4, HEAD_DIM)
q_rot, k_rot = apply_rotary_pos_emb(q, k, freqs_cos, freqs_sin)
assert q_rot.shape == q.shape
assert k_rot.shape == k.shape
print(f'freqs_cos shape: {freqs_cos.shape}')
print(f'q_rot shape:     {q_rot.shape}')
print(f'k_rot shape:     {k_rot.shape}')
print('✅ RoPE test passed')

## Part D — Grouped-Query Attention (GQA)

Standard Multi-Head Attention (MHA) uses equal numbers of Q, K, and V heads. **Grouped-Query Attention (GQA)** shares KV heads across multiple query heads:

- MiniMind uses **8 query heads** but only **4 KV heads**
- Each KV head is shared by 2 query heads (repeat factor = 2)
- This **halves the KV cache** during inference with minimal quality loss

The `repeat_kv` helper replicates KV heads to match the number of query heads before computing attention scores. The attention mechanism also includes:
- **QK normalization** via RMSNorm for training stability
- **Flash Attention** when available (fused CUDA kernel, faster + less memory)
- **Causal masking** to prevent attending to future tokens

In [ ]:
# === Grouped-Query Attention ===
def repeat_kv(x, n_rep):
    """Repeat KV heads to match number of query heads."""
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    # ============================================================
    # TODO: Expand KV heads by n_rep times
    #
    # Shape transformation: (bs, slen, num_kv_heads, head_dim)
    #                     → (bs, slen, num_kv_heads * n_rep, head_dim)
    #
    # Steps:
    #   1. Add a new dim: x[:, :, :, None, :]
    #      → shape (bs, slen, num_kv_heads, 1, head_dim)
    #   2. .expand(bs, slen, num_kv_heads, n_rep, head_dim)
    #   3. .reshape(bs, slen, num_kv_heads * n_rep, head_dim)
    # ============================================================

    # YOUR CODE HERE
    pass

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_key_value_heads = config.num_key_value_heads
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = self.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim

        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        # ============================================================
        # TODO: Implement the attention forward pass
        #
        # Steps:
        #   1. Project to Q, K, V:
        #      xq = self.q_proj(x),  xk = self.k_proj(x),  xv = self.v_proj(x)
        #   2. Reshape to (bsz, seq_len, n_heads, head_dim)
        #   3. Apply QK norms: xq = self.q_norm(xq), xk = self.k_norm(xk)
        #   4. Apply RoPE: xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        #      where cos, sin = position_embeddings
        #   5. Handle KV cache: if past_key_value is not None,
        #      concat past K/V along seq dim; set past_kv if use_cache
        #   6. Transpose to (bsz, n_heads, seq_len, head_dim):
        #      xq = xq.transpose(1,2)
        #      xk = repeat_kv(xk, self.n_rep).transpose(1,2)
        #      xv = repeat_kv(xv, self.n_rep).transpose(1,2)
        #   7. Compute attention:
        #      - If self.flash and seq_len > 1 and no cache:
        #          use F.scaled_dot_product_attention (is_causal=True)
        #      - Else: scores = (xq @ xk.T) / sqrt(head_dim)
        #              add causal mask (upper triangular = -inf)
        #              optionally add attention_mask
        #              output = softmax(scores) @ xv
        #   8. Reshape output: (bsz, seq_len, hidden_size)
        #   9. Apply output projection + residual dropout
        #  10. Return (output, past_kv)
        # ============================================================

        # YOUR CODE HERE
        pass

print('Attention class defined \u2713')


In [ ]:
# === ✅ Milestone 2A: Attention Output Shape ===
config = MiniMindConfig()
attn = Attention(config)
BATCH, SEQ_LEN = 2, 32
x = torch.randn(BATCH, SEQ_LEN, config.hidden_size)

freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=SEQ_LEN)
pos_emb = (freqs_cos, freqs_sin)

with torch.no_grad():
    out, _ = attn(x, pos_emb)

assert out.shape == (BATCH, SEQ_LEN, config.hidden_size), f'Shape: {out.shape}'
print(f'✅ Milestone 2A passed — Attention output shape: {out.shape}')
print(f'   Q heads: {config.num_attention_heads}, KV heads: {config.num_key_value_heads}')
print(f'   GQA repeat factor: {config.num_attention_heads // config.num_key_value_heads}')

## Part E — SwiGLU Feed-Forward Network

Instead of a simple ReLU-MLP, modern transformers use **SwiGLU** — a gated linear unit with the SiLU (Swish) activation:

$$\text{FFN}(x) = W_{\text{down}} \cdot (\text{SiLU}(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$ (smooth approximation to ReLU).

The gate and up projections form a "gated linear unit" — one branch controls the gate, the other provides the signal. This is more expressive than standard ReLU at similar computational cost, and is used in LLaMA, Gemma, and other modern architectures.

In [ ]:
# === SwiGLU Feed-Forward Network ===
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)

    def forward(self, x):
        # ============================================================
        # TODO: Implement SwiGLU feed-forward
        #
        # Formula: down_proj( SiLU(gate_proj(x)) * up_proj(x) )
        #
        # Steps:
        #   1. gate = self.gate_proj(x)         # shape: (B, T, intermediate_size)
        #   2. up   = self.up_proj(x)            # shape: (B, T, intermediate_size)
        #   3. Apply SiLU gate: F.silu(gate) * up
        #   4. Project back down: self.down_proj(...)
        #   5. Return the result
        # ============================================================

        # YOUR CODE HERE
        pass

print('FeedForward (SwiGLU) defined \u2713')


In [ ]:
# === ✅ Milestone 2B: Feed-Forward Shape ===
config = MiniMindConfig()
ffn = FeedForward(config)
x = torch.randn(2, 32, config.hidden_size)

with torch.no_grad():
    out = ffn(x)

assert out.shape == x.shape, f'Shape mismatch: {out.shape}'
print(f'✅ Milestone 2B passed — FFN output shape: {out.shape}')
print(f'   Expansion: {config.hidden_size} → {config.intermediate_size} → {config.hidden_size}')

## Part E½ — Mixture of Experts (MOE)

**Mixture of Experts (MOE)** replaces the standard feed-forward layer with multiple "expert" FFN sub-networks.
A learned **gating network** routes each token to the top-k experts, and outputs are combined by gating weights.

Key ideas:
- **Sparse activation**: only k of N experts run per token → more parameters without proportional compute
- **Load balancing**: an auxiliary loss encourages even distribution across experts
- **Drop-in replacement**: `MiniMindBlock` uses `MOEFeedForward` instead of `FeedForward` when `use_moe=True`


In [ ]:
# === Mixture of Experts Feed-Forward ===
class MOEFeedForward(nn.Module):
    """Mixture of Experts: route each token to top-k experts via a learned gate."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        self.experts = nn.ModuleList([
            FeedForward(config) for _ in range(config.num_experts)
        ])

    def forward(self, x):
        # ============================================================
        # TODO: Implement Mixture of Experts forward pass
        #
        # Steps:
        #   1. Flatten x to (batch*seq, hidden_dim):
        #      x_flat = x.view(-1, hidden_dim)
        #   2. Compute gating scores:
        #      scores = F.softmax(self.gate(x_flat), dim=-1)
        #   3. Select top-k experts per token:
        #      topk_weight, topk_idx = torch.topk(scores, k=..., dim=-1)
        #   4. Optionally normalize top-k weights:
        #      topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)
        #   5. Dispatch tokens to experts:
        #      for i, expert in enumerate(self.experts):
        #          mask = (topk_idx == i)
        #          if mask.any():
        #              token_idx = mask.any(dim=-1).nonzero().flatten()
        #              weight = topk_weight[mask].view(-1, 1)
        #              y.index_add_(0, token_idx, expert(x_flat[token_idx]) * weight)
        #   6. Compute auxiliary load-balancing loss (self.aux_loss)
        #   7. Reshape output back to (batch, seq, hidden_dim)
        # ============================================================

        # YOUR CODE HERE
        pass

print('MOEFeedForward defined ✓')


In [ ]:
# === ✅ Milestone 2B½: MOE Feed-Forward ===
moe_config = MiniMindConfig()
moe_config.use_moe = True
moe_ff = MOEFeedForward(moe_config)
x = torch.randn(2, 8, moe_config.hidden_size)
out = moe_ff(x)
assert out.shape == x.shape, f'MOE output shape mismatch: {out.shape} != {x.shape}'
print(f'✅ Milestone 2B½ passed — MOE output shape: {out.shape}')
print(f'  Experts: {moe_config.num_experts}, active per token: {moe_config.num_experts_per_tok}')
print(f'  Auxiliary loss: {moe_ff.aux_loss.item():.6f}')


## Part F — Transformer Block

Each transformer block applies two sub-layers with **pre-norm** residual connections:

1. **Pre-norm → Attention → Residual:** `x = x + Attention(RMSNorm(x))`
2. **Pre-norm → FFN → Residual:** `x = x + FFN(RMSNorm(x))`

**Pre-norm** (normalize *before* the sublayer) is more stable than post-norm (used in the original Transformer). It prevents gradient explosion in deep networks and is standard in all modern LLMs.

In [ ]:
# === Transformer Block ===
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        # ============================================================
        # TODO: Implement the transformer block with pre-norm residuals
        #
        # Sub-layer 1 — Self-Attention:
        #   residual = hidden_states
        #   norm_x   = self.input_layernorm(hidden_states)
        #   attn_out, present_key_value = self.self_attn(norm_x, position_embeddings,
        #                                                 past_key_value, use_cache, attention_mask)
        #   hidden_states = residual + attn_out       ← residual connection
        #
        # Sub-layer 2 — Feed-Forward:
        #   norm_x       = self.post_attention_layernorm(hidden_states)
        #   hidden_states = hidden_states + self.mlp(norm_x)  ← residual connection
        #
        # Return: (hidden_states, present_key_value)
        # ============================================================

        # YOUR CODE HERE
        pass

print('MiniMindBlock defined \u2713')


In [ ]:
# === Test Transformer Block ===
config = MiniMindConfig()
block = MiniMindBlock(layer_id=0, config=config)
x = torch.randn(2, 32, config.hidden_size)
freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=32)

with torch.no_grad():
    out, _ = block(x, (freqs_cos, freqs_sin))

assert out.shape == x.shape, f'Shape: {out.shape}'
print(f'Block output shape: {out.shape} ✓')

## Part G — Complete MiniMind Model

We stack N transformer blocks, add token embeddings and a language model head to form the complete model:

1. **Token Embedding** — maps token IDs to dense vectors (`vocab_size → hidden_size`)
2. **N × MiniMindBlock** — the transformer stack with attention + FFN
3. **Final RMSNorm** — normalize before the output projection
4. **LM Head** — projects hidden states back to vocabulary logits (`hidden_size → vocab_size`)

**Weight tying:** The embedding matrix is shared with the LM head output projection. This reduces parameters and often improves performance — the model uses the same representation space for input tokens and output predictions.

In [ ]:
# === Complete MiniMind Model ===
class MiniMindModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([
            MiniMindBlock(l, config) for l in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(
            dim=config.head_dim,
            end=config.max_position_embeddings,
            rope_base=config.rope_theta
        )
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        batch_size, seq_length = input_ids.shape
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0

        # ============================================================
        # TODO: Implement MiniMindModel forward pass
        #
        # Steps:
        #   1. Embed tokens and apply dropout:
        #      hidden_states = self.dropout(self.embed_tokens(input_ids))
        #   2. Slice position embeddings for current sequence:
        #      position_embeddings = (
        #          self.freqs_cos[start_pos : start_pos + seq_length],
        #          self.freqs_sin[start_pos : start_pos + seq_length]
        #      )
        #   3. Pass through each layer, collecting KV cache:
        #      presents = []
        #      for layer, past_kv in zip(self.layers, past_key_values):
        #          hidden_states, present = layer(hidden_states, position_embeddings,
        #                                         past_key_value=past_kv, use_cache=use_cache,
        #                                         attention_mask=attention_mask)
        #          presents.append(present)
        #   4. Apply final norm: hidden_states = self.norm(hidden_states)
        #   5. Compute aux_loss from MOE layers:
        #      aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], ...)
        #   6. Return (hidden_states, presents, aux_loss)
        # ============================================================

        # YOUR CODE HERE
        pass


class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        # Weight tying: share embedding and output projection weights
        self.model.embed_tokens.weight = self.lm_head.weight

    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        # ============================================================
        # TODO: Implement the causal LM forward pass
        #
        # Steps:
        #   1. Call self.model(...) to get hidden_states, past_key_values, and aux_loss
        #   2. Project to vocabulary: logits = self.lm_head(hidden_states)
        #   3. If labels is not None, compute cross-entropy loss:
        #      - Shift: shift_logits = logits[..., :-1, :].contiguous()
        #               shift_labels = labels[..., 1:].contiguous()
        #      - Loss: F.cross_entropy(shift_logits.view(-1, vocab_size),
        #                              shift_labels.view(-1), ignore_index=-100)
        #   4. Return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values, 'aux_loss': aux_loss}
        # ============================================================

        # YOUR CODE HERE
        pass

print('MiniMindForCausalLM defined \u2713')


In [ ]:
# === ✅ Milestone 2C: Complete Model & Parameter Count ===
config = MiniMindConfig()
model = MiniMindForCausalLM(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Milestone 2C passed — complete model built')
print(f'   Total parameters:     {total_params:>12,}')
print(f'   Trainable parameters: {trainable_params:>12,}')
print(f'   Model size (MB):      {total_params * 4 / 1024 / 1024:>12.1f}')

# Quick forward pass test
input_ids = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    outputs = model(input_ids)

assert outputs['logits'].shape == (2, 32, config.vocab_size), f'Logits shape: {outputs["logits"].shape}'
print(f'   Logits shape:         {outputs["logits"].shape}')

# Test with labels (loss computation)
labels = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    outputs = model(input_ids, labels=labels)

assert outputs['loss'] is not None, 'Loss should not be None when labels provided'
print(f'   Loss (random init):   {outputs["loss"].item():.4f}')
print(f'   Expected ~ln({config.vocab_size})={math.log(config.vocab_size):.2f} for random model')

In [ ]:
# === Parameter Breakdown by Layer ===
print('Parameter breakdown:')
print('-' * 55)
for name, param in model.named_parameters():
    if 'layers.0.' in name or 'embed' in name or 'lm_head' in name or 'norm.weight' == name.split('.')[-1] and 'layers' not in name:
        if 'layers.0.' in name:
            layer_name = name.replace('model.layers.0.', 'block.')
            count = param.numel()
            print(f'  {layer_name:45s} {count:>10,}  {list(param.shape)}')
        elif 'embed' in name:
            print(f'  {"embed_tokens":45s} {param.numel():>10,}  {list(param.shape)}')

# Per-block total
block_params = sum(p.numel() for n, p in model.named_parameters() if 'layers.0.' in n)
print(f'\n  Per-block total: {block_params:,} × {config.num_hidden_layers} layers = {block_params * config.num_hidden_layers:,}')
embed_params = model.model.embed_tokens.weight.numel()
print(f'  Embedding (tied with lm_head): {embed_params:,}')
print(f'  Final norm: {model.model.norm.weight.numel():,}')

## 🎯 Notebook 2 Summary

| Component | Key Concept |
|---|---|
| **MiniMindConfig** | Hyperparameters: 768 hidden, 8 layers, 8 Q / 4 KV heads |
| **RMSNorm** | Normalize by RMS (no mean subtraction), learnable scale |
| **RoPE** | Encode position via rotation matrices in 2D subspaces |
| **GQA Attention** | Share KV heads across Q head groups (4 KV → 8 Q) |
| **SwiGLU FFN** | Gated linear unit: down(SiLU(gate(x)) × up(x)) |
| **MOE FFN** | Route tokens to top-k of N experts via learned gate |
| **MiniMindBlock** | Pre-norm → Attention → Residual → Pre-norm → FFN/MOE → Residual |
| **MiniMindForCausalLM** | Full model with weight-tied embedding & LM head |

**Milestones completed:** 2A (Attention shape), 2B (FFN shape), 2B½ (MOE shape), 2C (complete model)

### Next Notebook
In Notebook 3, we build the **data pipeline** — loading and preprocessing text data for pretraining and supervised fine-tuning, with proper label masking.
